# Notebook 16 - Multimodal Agents Across Speech and Vision

This notebook shows how agent loops can delegate to modality-specific tools instead of pretending one giant prompt should do everything.


## What you will learn

- how to represent modality-specific tools
- how to route agent decisions by evidence type
- how to inspect traces for multimodal failure modes


In [ ]:
!pip install -q numpy pandas matplotlib pydantic
print("Installed notebook dependencies.")


In [ ]:
import json
import math
import random
import statistics
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-multimodal")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ROOT = ARTIFACT_ROOT / "16_multimodal_agents_across_speech_and_vision"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(7)
np.random.seed(7)



print("Cache dir:", CACHE_DIR)
print("Artifact root:", NOTEBOOK_ROOT.resolve())


## Step 1: Define the tools

Separate tools make the handoff between image, speech, and retrieval logic explicit.


In [ ]:
@dataclass
class ToolCall:
    tool_name: str
    reason: str
    payload: Dict[str, Any]

available_tools = [
    ToolCall("transcribe_audio", "Need words and timestamps from a clip", {"path": "meeting.wav"}),
    ToolCall("inspect_image", "Need chart evidence from screenshot", {"path": "chart.png"}),
    ToolCall("retrieve_page", "Need document page supporting a field", {"query": "invoice total"}),
]
pd.DataFrame([asdict(tool) for tool in available_tools])


## Step 2: Route by evidence type

The routing function below is intentionally simple; the main lesson is that modality routing should be explicit and inspectable.


In [ ]:
def route_tool(needs_audio=False, needs_image=False, needs_document=False):
    if needs_audio:
        return "transcribe_audio"
    if needs_image:
        return "inspect_image"
    if needs_document:
        return "retrieve_page"
    return "draft_answer"

agent_requests = pd.DataFrame([
    {"task": "summarize voice note", "tool": route_tool(needs_audio=True)},
    {"task": "read the chart", "tool": route_tool(needs_image=True)},
    {"task": "find the invoice total", "tool": route_tool(needs_document=True)},
])
agent_requests


## Step 3: Review the trace

Trace review is where agent design becomes engineering instead of folklore.


In [ ]:
trace = pd.DataFrame([
    {"step": 1, "decision": "transcribe_audio", "status": "ok"},
    {"step": 2, "decision": "retrieve_page", "status": "ok"},
    {"step": 3, "decision": "draft_answer", "status": "missing_page_box"},
])
trace


## Exercises and extensions

1. Add a visual tool that returns bounding boxes as part of its observation.
1. Create a trace-quality metric that penalizes unnecessary tool calls.
